# cvenv — component-by-component setup demo

One cell per component (`install` + `verify`). Use this as a template: copy the
single cell you need into any tutorial notebook. The GPU components
(`pytorch3d`, `mast3r`, `sam2`) need a GPU runtime; `science` runs anywhere.

> If a step changes **numpy**, restart the kernel/runtime once before continuing.

In [8]:
# Install cvenv itself (no heavy deps). Pin a tag for reproducibility.
!pip install -q "git+https://github.com/ribeiro-computer-vision/cvenv@v0.1.6"

import cvenv
print("cvenv", cvenv.__version__)
cvenv.PlatformManager().detect_platform()

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
cvenv 0.1.4


('Colab', '/content/')

In [9]:
# See what's available (with the teaching notes)
for c in cvenv.list_components():
    print(f"{c.name:<10} {c.summary}")
    print(f"           why: {c.teaching_note}\n")

mast3r     NAVER MASt3R matcher: clone repo, install deps, fetch checkpoint.
           why: MASt3R is used as a source checkout, not a pip package — the repo root must be on sys.path for 'import mast3r'. Its checkpoint is multi-GB, so the download resumes on failure. A local copy can be reused via checkpoint_dir= to skip the download.

opengl     PyOpenGL + system GL/GLUT dev libraries (for pyrender / rendering).
           why: Offscreen GL on a headless server needs a context: set PYOPENGL_PLATFORM=egl (GPU, fast) or osmesa (CPU, robust). The apt libs here (freeglut3-dev, libglew-dev, libsdl2-dev) only install on Debian/Ubuntu images; they're skipped elsewhere.

pytorch3d  Facebook PyTorch3D (differentiable 3D). Wheel if possible, else source build.
           why: The single hardest install here. A CUDA-matched prebuilt wheel is far faster than a source build. Always test 'import pytorch3d._C' — plain 'import pytorch3d' succeeds even when the compiled _C extension is broken. A whee

## Mount Google Drive
On Colab, cvenv saves wheels to /content/drive/MyDrive/cvenv_wheels/ by default.

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## science — base numpy-2 scientific stack
Runs anywhere. This is all you need for pure-numpy/scipy material (Kalman filters, Lie groups).

In [10]:
cvenv.get_component("science").install()
cvenv.get_component("science").verify()

✅ science: already installed — skipping.
✅ science: numpy 2.0.2, scipy + cv2 4.13.0 import OK


True

## pytorch3d — differentiable 3D (GPU)
Fastest with a prebuilt wheel matching this runtime's torch/CUDA/cp version.
Set `MY_WHEEL` to that URL, or leave `None` to try the official index / source build.

In [15]:
import glob, os
WHEEL = max(glob.glob("/content/drive/MyDrive/cvenv_wheels/pytorch3d-*.whl"),
            key=os.path.getmtime)
print("using:", WHEEL)

cvenv.get_component("pytorch3d").install(wheel_url=WHEEL)
cvenv.get_component("pytorch3d").verify()   # want: ✅ pytorch3d … (_C OK)

using: /content/drive/MyDrive/cvenv_wheels/pytorch3d-0.7.8-cp312-cp312-linux_x86_64.whl
▶️  pytorch3d: installing…
$ /usr/bin/python3 -m pip install numpy>=2.0,<2.1
$ /usr/bin/python3 -m pip install iopath
Installing PyTorch3D from wheel: pytorch3d-0.7.8-cp312-cp312-linux_x86_64.whl
$ /usr/bin/python3 -m pip install --force-reinstall --no-deps /content/drive/MyDrive/cvenv_wheels/pytorch3d-0.7.8-cp312-cp312-linux_x86_64.whl
✅ installed from provided wheel.
✅ pytorch3d 0.7.8 (_C OK)


True

## mast3r — NAVER matcher (GPU)
Clones the repo, installs deps, downloads the checkpoint. Pass `checkpoint_dir=` to reuse a local copy.

In [16]:
cvenv.get_component("mast3r").install()   # add checkpoint_dir="..." to skip the download
cvenv.get_component("mast3r").verify()

▶️  mast3r: installing…
$ git clone --recursive https://github.com/naver/mast3r /content/mast3r
$ /usr/bin/python3 -m pip install -r /content/mast3r/requirements.txt
$ /usr/bin/python3 -m pip install -r /content/mast3r/dust3r/requirements.txt
$ /usr/bin/python3 -m pip install -r /content/mast3r/dust3r/requirements_optional.txt
⬇️  downloading MASt3R checkpoint to /content/mast3r/checkpoints/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth
Warning, cannot find cuda-compiled version of RoPE2D, using a slow pytorch version instead
✅ mast3r importable (repo on sys.path, checkpoint present)


True

## sam2 — Meta Segment Anything 2 (GPU)
pip-installs SAM2 and downloads its checkpoint. Store it under a persistent path via `checkpoint_dir=`.

In [17]:
cvenv.get_component("sam2").install(checkpoint_dir="checkpoints")
cvenv.get_component("sam2").verify(checkpoint_dir="checkpoints")

▶️  sam2: installing…
$ /usr/bin/python3 -m pip install --no-cache-dir git+https://github.com/facebookresearch/sam2.git
⬇️  downloading SAM2 checkpoint to checkpoints/sam2.1_hiera_large.pt
✅ sam2 importable; checkpoint present (checkpoints/sam2.1_hiera_large.pt)


True